In [ ]:
import afmlevel.afmlevel_functions as afml
import os
import numpy as np
from PIL import Image
from sklearn.metrics import mean_squared_error
from skimage.metrics import peak_signal_noise_ratio
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from pytorch_msssim import MS_SSIM
import torch

In [ ]:
def znorm(image):
    mean = np.mean(image)
    std_dev = np.std(image)
    imnorm_z = (image - mean) / std_dev
    return imnorm_z


def minmaxnorm(image):
    if isinstance(image, torch.Tensor):
        im0 = image - image.min()
        imnorm = im0 / im0.max()
        return imnorm
    elif isinstance(image, np.ndarray):
        im0 = image - np.min(image)
        imnorm = im0 / np.max(im0)
        return imnorm


def xyplanefit(imarray, polyx, polyy):
    mc = np.mean(imarray, axis=0)
    x = np.arange(0, len(mc), 1)
    p = np.polyfit(x, mc, polyx)
    p = np.poly1d(p)
    pvals = np.polyval(p, x)
    r = imarray - pvals[np.newaxis, :]

    mr = np.mean(r, axis=1)
    y = np.arange(0, len(mr), 1)
    p = np.polyfit(y, mr, polyy)
    p = np.poly1d(p)
    pvals = np.polyval(p, y)
    r = r - pvals[:, np.newaxis]

    return r


def MS_SSIMloss(predicted, groundtruth, device="cpu"):
    if not isinstance(predicted, torch.Tensor):
        predicted = torch.from_numpy(predicted)
    if not isinstance(groundtruth, torch.Tensor):
        groundtruth = torch.from_numpy(groundtruth)

    predicted = predicted.float().to(device)
    groundtruth = groundtruth.float().to(device)
    predicted = predicted.unsqueeze(0).unsqueeze(0)
    groundtruth = groundtruth.unsqueeze(0).unsqueeze(0)

    MS_SSIMfn = MS_SSIM(data_range=1, size_average=True, channel=1)
    loss = 1 - MS_SSIMfn(predicted, groundtruth)
    return loss.item()

## Configuration

In [ ]:

saveimgs = False
label = "BGfunctionpixsplit_apply3x_Alltestdata"

base_path = r"C:\Users\ggjh246\OneDrive - University of Leeds\Code\ApplyModels_Eddie"
test_filename = "data"
test_path = os.path.join(base_path, test_filename)

AFM = np.load(r"C:\Users\ggjh246\OneDrive - University of Leeds\Code\afmlevel\src\afmlevel\lutAFM.npy")
AFM = ListedColormap(AFM)

MSEscore = False
SSIMscore = True
PSNRscore = True

if saveimgs:
    os.makedirs(f"{label}_images", exist_ok=True)


# Load models
bg_model_path = r"C:\Users\ggjh246\OneDrive - University of Leeds\Code\ApplyModels_Eddie\unetmodel_Aire_BG_60eps_b32_d0_1f9_f9_7l_MSE_0line_train34_epoch59.pth"
mask_model_path = r"C:\Users\ggjh246\OneDrive - University of Leeds\Code\ApplyModels_Eddie\unetmodel_Aire_Mask_80eps_b32_d0_1f7_f7_7l_line2_train34_otsublur1_epoch69.pth"

## Load Image IDs

In [ ]:

imageidlist = []

for filename in os.listdir(test_path):
    if filename.endswith(".tiff") and not filename.endswith("_levelled.tiff"):
        imageidlist.append(filename[:-5])  # remove .tiff

imageidlist[:10]  # preview


## Processing Loop

In [ ]:
MSElist = []
SSIMlist = []
PSNRlist = []

for i, image_id in enumerate(imageidlist):

    print(f"Processing image {i+1}/{len(imageidlist)}: {image_id}")

    # --- Load image ---
    image = np.array(Image.open(os.path.join(test_path, f"{image_id}.tiff")))
    
    # --- Apply background model (1x, 2x, 3x) ---
    model_bg, model_levarray = afml.applymodel_bg_pixelsplit_all(image, 3, bg_model_path)
    model_levnorm = znorm(model_levarray).astype(np.float32)

    model_bg2, model_levarray2 = afml.applymodel_bg_pixelsplit_all(model_levarray, 3, bg_model_path)
    model_levnorm2 = znorm(model_levarray2)

    model_bg3, model_levarray3 = afml.applymodel_bg_pixelsplit_all(model_levarray2, 3, bg_model_path)
    model_levnorm3 = znorm(model_levarray3)


## Plotting

In [ ]:

plt.figure(figsize=(18, 4))
plt.subplot(1, 3, 1)
plt.title("Original")
plt.imshow(image, cmap=AFM)


plt.subplot(1, 3, 3)
plt.title("Model Levelled (first pass)")
plt.imshow(model_levnorm, cmap=AFM)

plt.show()
